In [3]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

In [4]:
def polygon_area(points):

    x = np.array([p[0] for p in points])
    y = np.array([p[1] for p in points])

    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

In [5]:
def polygon_centroid(points):

    x = np.array([p[0] for p in points])
    y = np.array([p[1] for p in points])

    centroid_x = np.mean(x)
    centroid_y = np.mean(y)

    return centroid_x, centroid_y

In [19]:
def extract_rooms(svg_path):

    tree = ET.parse(svg_path)
    root = tree.getroot()

    rooms = []

    for elem in root.iter():

        if "class" in elem.attrib:

            class_name = elem.attrib["class"]

            if class_name.startswith("Space"):

                room_type = class_name.replace("Space ", "")

                polygon = elem.find(".//{http://www.w3.org/2000/svg}polygon")

                if polygon is None:
                    continue

                points_str = polygon.attrib["points"]

                points = []

                for p in points_str.split():
                    x, y = p.split(",")
                    points.append((float(x), float(y)))

                area = polygon_area(points)
                centroid = polygon_centroid(points)

                rooms.append({
                    "room_type": room_type,
                    "area": area,
                    "centroid_x": centroid[0],
                    "centroid_y": centroid[1],
                    "num_vertices": len(points),
                    "points":points,
                })

    return rooms

In [21]:
df = pd.read_csv("cubicasa_plan_index.csv")

In [22]:
sample_svg = df.iloc[0]["svg_path"]

rooms = extract_rooms(sample_svg)

for r in rooms:
    print(r)

{'room_type': 'Outdoor Terrace', 'area': 248496.02960000024, 'centroid_x': 1260.5033333333333, 'centroid_y': 387.0833333333333, 'num_vertices': 6, 'points': [(1030.81, 507.29), (1030.81, 146.67), (1719.89, 146.67), (1719.89, 507.29), (1030.81, 507.29), (1030.81, 507.29)]}
{'room_type': 'Entry Lobby', 'area': 54825.5027999999, 'centroid_x': 1414.4666666666665, 'centroid_y': 1211.0933333333332, 'num_vertices': 9, 'points': [(1469.34, 1137.48), (1469.34, 1148.48), (1530.77, 1148.48), (1530.77, 1293.83), (1463.05, 1293.83), (1463.05, 1320.35), (1234.77, 1320.35), (1234.77, 1118.52), (1334.34, 1118.52)]}
{'room_type': 'Kitchen', 'area': 67064.8975999998, 'centroid_x': 1576.1033333333335, 'centroid_y': 1320.6066666666666, 'num_vertices': 6, 'points': [(1712.49, 1478.05), (1474.05, 1478.05), (1474.05, 1346.29), (1541.77, 1346.29), (1541.77, 1137.48), (1712.49, 1137.48)]}
{'room_type': 'LivingRoom', 'area': 221476.7801000001, 'centroid_x': 1482.9166666666667, 'centroid_y': 934.7633333333333, '

In [23]:
print("Total rooms:", len(rooms))

Total rooms: 10


In [24]:
all_rooms = []

for _, row in df.iterrows():

    try:

        rooms = extract_rooms(row["svg_path"])

        for r in rooms:

            r["plan_id"] = row["plan_id"]

            all_rooms.append(r)

    except Exception as e:

        print("Error:", row["svg_path"])

In [25]:
rooms_df = pd.DataFrame(all_rooms)

rooms_df.head()

,room_type,area,centroid_x,centroid_y,num_vertices,points,plan_id
0,Outdoor Terrace,248496.0296,1260.503333,387.083333,6,"[(1030.81, 507.29), (1030.81, 146.67), (1719.8...",6044
1,Entry Lobby,54825.5028,1414.466667,1211.093333,9,"[(1469.34, 1137.48), (1469.34, 1148.48), (1530...",6044
2,Kitchen,67064.8976,1576.103333,1320.606667,6,"[(1712.49, 1478.05), (1474.05, 1478.05), (1474...",6044
3,LivingRoom,221476.7801,1482.916667,934.763333,6,"[(1712.49, 1137.48), (1469.34, 1137.48), (1334...",6044
4,Utility Laundry,20826.7515,1121.873333,1377.933333,6,"[(1190.03, 1478.05), (1051.17, 1478.05), (1051...",6044


In [26]:
rooms_df.to_csv("rooms_dataset.csv", index=False)